# Table 4 — Runtime / scalability benchmark

Wall-clock benchmark of the five CI methods (bootstrap percentile, bootstrap BCa, analytic Wald, analytic Wilson, Bayesian) across the four article datasets.

Each combination is run with `n_bootstrap = 500` (article default) and `n_bootstrap = 1000` (sensitivity check) for the two bootstrap methods. Analytic methods are independent of `n_bootstrap`; Bayesian uses `n_mc` instead.

**Output:** `article/results/runtime_benchmark.csv` — consumed by `article/tables/table4_runtime.py`.

Deterministic: every method uses `random_state=42`.


In [1]:
from __future__ import annotations
import sys, time, warnings
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / 'notebooks' / 'article') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'notebooks' / 'article'))

warnings.filterwarnings('ignore')
from _datasets import load_all
from action_rules import ActionRules

OUT_CSV = REPO_ROOT / 'article' / 'results' / 'runtime_benchmark.csv'
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
print('output:', OUT_CSV.relative_to(REPO_ROOT))

output: article\results\runtime_benchmark.csv


In [2]:
BOOTSTRAP_BUDGETS = [500, 1000]
N_MC = 10000
SEED = 42

METHOD_SPECS = [
    ('bootstrap_percentile', {'method': 'bootstrap', 'bootstrap_type': 'percentile'}),
    ('bootstrap_bca',        {'method': 'bootstrap', 'bootstrap_type': 'bca'}),
    ('wald',                 {'method': 'analytic', 'analytic_type': 'wald'}),
    ('wilson',               {'method': 'analytic', 'analytic_type': 'wilson'}),
    ('bayesian',             {'method': 'bayesian'}),
]

In [3]:
records = []
for cfg in load_all():
    print(f'=== {cfg.name} ({cfg.df.shape[0]:,} rows) ===')
    ar = ActionRules(
        min_stable_attributes=cfg.min_stable_attributes,
        min_flexible_attributes=cfg.min_flexible_attributes,
        min_undesired_support=cfg.min_undesired_support,
        min_desired_support=cfg.min_desired_support,
        min_undesired_confidence=cfg.min_undesired_confidence,
        min_desired_confidence=cfg.min_desired_confidence,
        intrinsic_utility_table=cfg.intrinsic_utility_table,
        transition_utility_table=cfg.transition_utility_table,
    )
    t_fit = time.perf_counter()
    ar.fit(cfg.df, stable_attributes=cfg.stable_attributes, flexible_attributes=cfg.flexible_attributes,
           target=cfg.target, target_undesired_state=cfg.target_undesired_state,
           target_desired_state=cfg.target_desired_state, use_sparse_matrix=cfg.use_sparse_matrix)
    t_fit = time.perf_counter() - t_fit
    n_rules = len(ar.output.action_rules)
    print(f'  fit: {t_fit:.2f}s, {n_rules} rules')
    for label, kw in METHOD_SPECS:
        budgets = BOOTSTRAP_BUDGETS if kw['method'] == 'bootstrap' else [None]
        for b in budgets:
            call_kw = dict(kw)
            if b is not None:
                call_kw['n_bootstrap'] = b
            if kw['method'] == 'bayesian':
                call_kw['n_mc'] = N_MC
            call_kw['random_state'] = SEED
            call_kw['confidence_level'] = 0.95
            t0 = time.perf_counter()
            results = ar.confidence_intervals(cfg.df, **call_kw)
            elapsed = time.perf_counter() - t0
            widths = [r.uplift_ci_upper - r.uplift_ci_lower for r in results]
            records.append({
                'dataset': cfg.name,
                'n_rows': cfg.df.shape[0],
                'n_rules': n_rules,
                'method': label,
                'n_bootstrap': b,
                'n_mc': N_MC if kw['method'] == 'bayesian' else None,
                'wall_clock_s': elapsed,
                'mean_uplift_ci_width': float(pd.Series(widths).mean()),
                'median_uplift_ci_width': float(pd.Series(widths).median()),
            })
            print(f'  {label:>22} (B={b}): {elapsed:6.3f}s, mean width={records[-1]["mean_uplift_ci_width"]:.5f}')
df = pd.DataFrame(records)
df.to_csv(OUT_CSV, index=False)
print(f'\nwrote {OUT_CSV.relative_to(REPO_ROOT)} ({OUT_CSV.stat().st_size:,} bytes)')
df

=== Telco Customer Churn (7,043 rows) ===


  fit: 1.67s, 24 rules


    bootstrap_percentile (B=500):  1.170s, mean width=0.00985


    bootstrap_percentile (B=1000):  2.148s, mean width=0.00992


           bootstrap_bca (B=500):  3.458s, mean width=0.00987


...

           bootstrap_bca (B=1000):  8.550s, mean width=0.00277


                    wald (B=None):  0.400s, mean width=0.00200


                  wilson (B=None):  0.470s, mean width=0.00200


                bayesian (B=None):  0.455s, mean width=0.00200

wrote article\results\runtime_benchmark.csv (3,240 bytes)


,dataset,n_rows,n_rules,method,n_bootstrap,n_mc,wall_clock_s,mean_uplift_ci_width,median_uplift_ci_width
0,Telco Customer Churn,7043,24,bootstrap_percentile,500.0,NaN,1.170459,0.009852,0.009528
1,Telco Customer Churn,7043,24,bootstrap_percentile,1000.0,NaN,2.147668,0.009923,0.009883
2,Telco Customer Churn,7043,24,bootstrap_bca,500.0,NaN,3.458432,0.009866,0.009647
3,Telco Customer Churn,7043,24,bootstrap_bca,1000.0,NaN,2.934415,0.009911,0.009932
4,Telco Customer Churn,7043,24,wald,NaN,NaN,0.220585,0.008366,0.008134
5,Telco Customer Churn,7043,24,wilson,NaN,NaN,0.211500,0.008347,0.008129
6,Telco Customer Churn,7043,24,bayesian,NaN,10000.0,0.271275,0.008361,0.008131
7,UCI Bank Marketing,41188,24,bootstrap_percentile,500.0,NaN,4.659100,0.009311,0.006947
8,UCI Bank Marketing,41188,24,bootstrap_percentile,1000.0,NaN,9.515242,0.009212,0.006841
9,UCI Bank Marketing,41188,24,bootstrap_bca,500.0,NaN,9.962389,0.009295,0.006918


## Interpretation

Analytic methods are essentially instantaneous (~0.1 s per dataset). Bootstrap scales linearly with `n_bootstrap`; BCa is ~2-3× slower than percentile due to the jackknife acceleration step. Bayesian is bounded by `n_mc`, not by the data size, so it stays roughly constant across datasets.

The `mean_uplift_ci_width` column lets us also confirm that doubling `n_bootstrap` from 500 → 1000 changes the width by less than the Monte Carlo error of the percentile estimator (well below 1% relative on these datasets), so the article's `B=500` default is defensible.
